In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

In [2]:
# 1. 读取数据
data = pd.read_csv('/Users/hezixin/Downloads/2023上机程序/data/trial.csv',header=0)
data.head()

,Sector_score,LOCATION_ID,PARA_A,SCORE_A,PARA_B,SCORE_B,TOTAL,numbers,Marks,Money_Value,MONEY_Marks,District,Loss,LOSS_SCORE,History,History_score,Score,Risk
0,3.89,23,4.18,6,2.50,2,6.68,5.0,2,3.38,2,2,0,2,0,2,2.4,1
1,3.89,6,0.00,2,4.83,2,4.83,5.0,2,0.94,2,2,0,2,0,2,2.0,0
2,3.89,6,0.51,2,0.23,2,0.74,5.0,2,0.00,2,2,0,2,0,2,2.0,0
3,3.89,6,0.00,2,10.80,6,10.80,6.0,6,11.75,6,2,0,2,0,2,4.4,1
4,3.89,6,0.00,2,0.08,2,0.08,5.0,2,0.00,2,2,0,2,0,2,2.0,0


In [5]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 776 entries, 0 to 775
Data columns (total 18 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Sector_score   776 non-null    float64
 1   LOCATION_ID    776 non-null    object 
 2   PARA_A         776 non-null    float64
 3   SCORE_A        776 non-null    int64  
 4   PARA_B         776 non-null    float64
 5   SCORE_B        776 non-null    int64  
 6   TOTAL          776 non-null    float64
 7   numbers        776 non-null    float64
 8   Marks          776 non-null    int64  
 9   Money_Value    775 non-null    float64
 10  MONEY_Marks    776 non-null    int64  
 11  District       776 non-null    int64  
 12  Loss           776 non-null    int64  
 13  LOSS_SCORE     776 non-null    int64  
 14  History        776 non-null    int64  
 15  History_score  776 non-null    int64  
 16  Score          776 non-null    float64
 17  Risk           776 non-null    int64  
dtypes: float64

In [7]:
# 假设最后一列是标签列，其余为特征
X = data.iloc[:, :-1]
y = data.iloc[:, -1]

In [17]:
# 尝试转换 'Feature1' 列为浮点数
# 使用pd.to_numeric()转换，参数errors='coerce'会将无法转换的值设置为NaN
X['LOCATION_ID'] = pd.to_numeric(X['LOCATION_ID'], errors='coerce')

# 现在，将NaN值填充为0或选择其他适当的填充策略
X['LOCATION_ID'].fillna(0)

0      23.0
1       6.0
2       6.0
3       6.0
4       6.0
       ... 
771     9.0
772    16.0
773    14.0
774    18.0
775    15.0
Name: LOCATION_ID, Length: 776, dtype: float64

In [29]:
X.info()
X.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 776 entries, 0 to 775
Data columns (total 17 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Sector_score   776 non-null    float64
 1   LOCATION_ID    776 non-null    float64
 2   PARA_A         776 non-null    float64
 3   SCORE_A        776 non-null    int64  
 4   PARA_B         776 non-null    float64
 5   SCORE_B        776 non-null    int64  
 6   TOTAL          776 non-null    float64
 7   numbers        776 non-null    float64
 8   Marks          776 non-null    int64  
 9   Money_Value    775 non-null    float64
 10  MONEY_Marks    776 non-null    int64  
 11  District       776 non-null    int64  
 12  Loss           776 non-null    int64  
 13  LOSS_SCORE     776 non-null    int64  
 14  History        776 non-null    int64  
 15  History_score  776 non-null    int64  
 16  Score          776 non-null    float64
dtypes: float64(8), int64(9)
memory usage: 103.2 KB


,Sector_score,LOCATION_ID,PARA_A,SCORE_A,PARA_B,SCORE_B,TOTAL,numbers,Marks,Money_Value,MONEY_Marks,District,Loss,LOSS_SCORE,History,History_score,Score
0,3.89,23.0,4.18,6,2.50,2,6.68,5.0,2,3.38,2,2,0,2,0,2,2.4
1,3.89,6.0,0.00,2,4.83,2,4.83,5.0,2,0.94,2,2,0,2,0,2,2.0
2,3.89,6.0,0.51,2,0.23,2,0.74,5.0,2,0.00,2,2,0,2,0,2,2.0
3,3.89,6.0,0.00,2,10.80,6,10.80,6.0,6,11.75,6,2,0,2,0,2,4.4
4,3.89,6.0,0.00,2,0.08,2,0.08,5.0,2,0.00,2,2,0,2,0,2,2.0


In [23]:
# 划分数据集
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

In [37]:
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
import numpy as np

In [57]:
# 特征缩放
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [59]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint

# 定义参数分布
param_dist = {
    'kneighborsclassifier__n_neighbors': randint(1, 30),
    'kneighborsclassifier__weights': ['uniform', 'distance'],
    'kneighborsclassifier__algorithm': ['auto', 'ball_tree', 'kd_tree', 'brute']
}

# 创建 Pipeline，包括 Imputer 和 KNN
pipeline = make_pipeline(SimpleImputer(strategy='mean'), KNeighborsClassifier())

# 创建 RandomizedSearchCV 实例
random_search = RandomizedSearchCV(pipeline, param_distributions=param_dist, n_iter=100, cv=5, scoring='accuracy', random_state=42)

# 训练模型
random_search.fit(X_train, y_train)

# 输出最佳参数和对应的准确率
print("Best parameters:", random_search.best_params_)
print("Best cross-validation accuracy: {:.2f}".format(random_search.best_score_))

# 使用最佳参数预测测试集
y_pred = random_search.predict(X_test)
accuracy = (y_pred == y_test).mean()
print(f"Test set accuracy: {accuracy:.2f}")

Best parameters: {'kneighborsclassifier__algorithm': 'auto', 'kneighborsclassifier__n_neighbors': 2, 'kneighborsclassifier__weights': 'distance'}
Best cross-validation accuracy: 0.95
Test set accuracy: 0.97


In [63]:
from sklearn.model_selection import RandomizedSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from scipy.stats import randint

# 定义参数分布
param_dist = {
    'logisticregression__C': [0.001, 0.01, 0.1, 1, 10, 100],  # 正则化参数
    'logisticregression__penalty': ['l1', 'l2'],  # 正则化类型
    'logisticregression__solver': ['liblinear', 'saga'],  # 优化算法
    'logisticregression__max_iter': [100, 200, 300]  # 最大迭代次数
}

# 创建 Pipeline，包括 Imputer 和 LogisticRegression
pipeline = make_pipeline(SimpleImputer(strategy='mean'), LogisticRegression(random_state=42))

# 创建 RandomizedSearchCV 实例
random_search = RandomizedSearchCV(pipeline, param_distributions=param_dist, n_iter=100, cv=5, scoring='accuracy', random_state=42)

# 训练模型
random_search.fit(X_train, y_train)

# 输出最佳参数和对应的准确率
print("Best parameters:", random_search.best_params_)
print("Best cross-validation accuracy: {:.2f}".format(random_search.best_score_))

# 使用最佳参数预测测试集
y_pred = random_search.predict(X_test)
accuracy = (y_pred == y_test).mean()
print(f"Test set accuracy: {accuracy:.2f}")

/opt/anaconda3/lib/python3.12/site-packages/sklearn/model_selection/_search.py:320: UserWarning: The total space of parameters 72 is smaller than n_iter=100. Running 72 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/linear_model/_sag.py:349: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/linear_model/_sag.py:349: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/linear_model/_sag.py:349: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/linear_model/_sag.py:349: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages

Best parameters: {'logisticregression__solver': 'liblinear', 'logisticregression__penalty': 'l2', 'logisticregression__max_iter': 100, 'logisticregression__C': 100}
Best cross-validation accuracy: 0.99
Test set accuracy: 1.00


/opt/anaconda3/lib/python3.12/site-packages/sklearn/linear_model/_sag.py:349: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/linear_model/_sag.py:349: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


In [65]:
from sklearn.svm import SVC
from sklearn.model_selection import RandomizedSearchCV

# 定义参数分布
param_dist_svc = {
    'svc__C': [0.1, 1, 10, 100],  # 正则化参数
    'svc__gamma': [0.001, 0.01, 0.1, 1],  # 核函数的系数
    'svc__kernel': ['linear', 'rbf', 'poly'],  # 核函数类型
    'svc__class_weight': ['balanced', None]  # 类别权重
}

# 创建 Pipeline，包括 Imputer 和 SVC
pipeline_svc = make_pipeline(SimpleImputer(strategy='mean'), SVC())

# 创建 RandomizedSearchCV 实例
random_search_svc = RandomizedSearchCV(pipeline_svc, param_distributions=param_dist_svc, n_iter=100, cv=5, scoring='accuracy', random_state=42)

# 训练模型
random_search_svc.fit(X_train, y_train)

# 输出最佳参数和对应的准确率
print("Best parameters for SVC:", random_search_svc.best_params_)
print("Best cross-validation accuracy for SVC: {:.2f}".format(random_search_svc.best_score_))

# 使用最佳参数预测测试集
y_pred_svc = random_search_svc.predict(X_test)
accuracy_svc = (y_pred_svc == y_test).mean()
print(f"Test set accuracy for SVC: {accuracy_svc:.2f}")

/opt/anaconda3/lib/python3.12/site-packages/sklearn/model_selection/_search.py:320: UserWarning: The total space of parameters 96 is smaller than n_iter=100. Running 96 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


Best parameters for SVC: {'svc__kernel': 'linear', 'svc__gamma': 0.001, 'svc__class_weight': 'balanced', 'svc__C': 1}
Best cross-validation accuracy for SVC: 1.00
Test set accuracy for SVC: 1.00


In [67]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV

# 定义参数分布
param_dist_rf = {
    'randomforestclassifier__n_estimators': [100, 200, 300],  # 树的数量
    'randomforestclassifier__max_depth': [None, 10, 20, 30],  # 树的最大深度
    'randomforestclassifier__min_samples_split': [2, 5, 10],  # 分割内部节点所需的最小样本数
    'randomforestclassifier__min_samples_leaf': [1, 2, 4],  # 叶节点所需的最小样本数
    'randomforestclassifier__max_features': ['auto', 'sqrt', 'log2']  # 寻找最佳分割时要考虑的特征数量
}

# 创建 Pipeline，包括 Imputer 和 RandomForestClassifier
pipeline_rf = make_pipeline(SimpleImputer(strategy='mean'), RandomForestClassifier())

# 创建 RandomizedSearchCV 实例
random_search_rf = RandomizedSearchCV(pipeline_rf, param_distributions=param_dist_rf, n_iter=100, cv=5, scoring='accuracy', random_state=42)

# 训练模型
random_search_rf.fit(X_train, y_train)

# 输出最佳参数和对应的准确率
print("Best parameters for Random Forest:", random_search_rf.best_params_)
print("Best cross-validation accuracy for Random Forest: {:.2f}".format(random_search_rf.best_score_))

# 使用最佳参数预测测试集
y_pred_rf = random_search_rf.predict(X_test)
accuracy_rf = (y_pred_rf == y_test).mean()
print(f"Test set accuracy for Random Forest: {accuracy_rf:.2f}")

Best parameters for Random Forest: {'randomforestclassifier__n_estimators': 100, 'randomforestclassifier__min_samples_split': 10, 'randomforestclassifier__min_samples_leaf': 4, 'randomforestclassifier__max_features': 'sqrt', 'randomforestclassifier__max_depth': 10}
Best cross-validation accuracy for Random Forest: 1.00
Test set accuracy for Random Forest: 1.00


/opt/anaconda3/lib/python3.12/site-packages/sklearn/model_selection/_validation.py:540: FitFailedWarning: 
200 fits failed out of a total of 500.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
200 fits failed with the following error:
Traceback (most recent call last):
  File "/opt/anaconda3/lib/python3.12/site-packages/sklearn/model_selection/_validation.py", line 888, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "/opt/anaconda3/lib/python3.12/site-packages/sklearn/base.py", line 1473, in wrapper
    return fit_method(estimator, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/anaconda3/lib/python3.12/site-packages/sklearn/pipeline.py", line 473, in fit
    self._final_estima